# Loader Node — Unit Tests

Step-by-step testing of `loader_node` against real inputs from `data/inputs/`.
Run each cell independently to inspect intermediate outputs and logs.

**Inputs the Loader reads from the graph state:**
- `spec_doc_path`  → 3GPP markdown, parsed into `## ` sections.
- `rules_bank`     → pre-parsed JSON; iterated rule-by-rule downstream.
- `legacy_openapi` → optional pre-parsed YAML/JSON; when present the Loader
  seeds `final_openapi` from it; otherwise an empty skeleton is used
  (with `info` populated from `rules_bank.metadata` when possible).

**Pre-requisites:**
1. `data/inputs/3gpp/28532-i00.md`
2. `data/inputs/rules/rules_bank_*.json`
3. `data/inputs/legacy/TS28532_ProvMnS.yaml` (optional)

In [ ]:
# Step 1 — Imports and path setup
#
# Test fixtures (which spec, rules, legacy to use) live in
# openapi_generator.config.paths, so the notebook stays free of glob magic
# and ad-hoc strings. Edit paths.py to switch fixtures.

import json
from pathlib import Path

import yaml

from openapi_generator.config import get_logger
from openapi_generator.config.paths import (
    ROOT,
    TEST_LEGACY_PATH,
    TEST_RULES_PATH,
    TEST_SPEC_PATH,
)
from openapi_generator.nodes.loader import loader_node

logger = get_logger(__name__)

REPO_ROOT   = Path(ROOT).resolve()
SPEC_PATH   = Path(TEST_SPEC_PATH).resolve()
RULES_PATH  = Path(TEST_RULES_PATH).resolve()
LEGACY_PATH = Path(TEST_LEGACY_PATH).resolve()

logger.info(f"Repo root  : {REPO_ROOT}")
logger.info(f"Spec path  : {SPEC_PATH}")
logger.info(f"Rules path : {RULES_PATH}")
logger.info(f"Legacy path: {LEGACY_PATH if LEGACY_PATH.is_file() else '(not present)'}")

assert SPEC_PATH.is_file(),  f"Spec not found at {SPEC_PATH}"
assert RULES_PATH.is_file(), f"Rules not found at {RULES_PATH}"
# Legacy is optional — used only by the 'with legacy' scenario below.

In [ ]:
# Step 2 — Pre-parse rules_bank and legacy_openapi.
# In production, the CLI (main.py) does this and feeds the dicts to the graph.
# Here we mirror that so the Loader only sees already-parsed objects.

with open(RULES_PATH, "r", encoding="utf-8") as f:
    rules_bank = json.load(f)

legacy_openapi = None
if LEGACY_PATH.is_file():
    with open(LEGACY_PATH, "r", encoding="utf-8") as f:
        legacy_openapi = yaml.safe_load(f)

logger.info(f"rules_bank   : {len(rules_bank.get('rules', []))} rule(s)")
logger.info(f"  metadata   : {list((rules_bank.get('metadata') or {}).keys())}")
logger.info(f"legacy_openapi present: {legacy_openapi is not None}")
if legacy_openapi:
    logger.info(f"  paths      : {len(legacy_openapi.get('paths') or {})}")
    logger.info(f"  schemas    : {len((legacy_openapi.get('components') or {}).get('schemas') or {})}")
    logger.info(f"  info.title : {(legacy_openapi.get('info') or {}).get('title')!r}")

## Scenario A — all three inputs present (spec + rules + legacy)

Loader must:
1. Parse 3GPP sections.
2. Seed `final_openapi` from the legacy YAML (deep copy).
3. Forward `rules_bank` untouched (downstream nodes iterate `rules_bank["rules"]`).
4. Initialize loop counters.

In [3]:
# Step 3 — Run loader with all three inputs

state_A = {
    "spec_doc_path": str(SPEC_PATH),
    "rules_bank": rules_bank,
    "legacy_openapi": legacy_openapi,
}
out_A = loader_node(state_A)
logger.info(f"Output keys: {sorted(out_A.keys())}")

2026-05-25 21:53:53 [INFO] openapi_generator.nodes.loader: Loader → parsed 696 sections from 28532-i00.md (excluded 1 symbolic-title section(s))
2026-05-25 21:53:53 [INFO] openapi_generator.nodes.loader: Loader → rules_bank: 240 rule(s); legacy_openapi: present
2026-05-25 21:53:53 [INFO] openapi_generator.nodes.loader: Loader → seeding final_openapi from legacy (1 path(s), 15 schema(s))
2026-05-25 21:53:53 [INFO] __main__: Output keys: ['current_op_idx', 'final_openapi', 'op_iteration_count', 'parsed_spec_sections', 'validated_fragments_by_op', 'validation_errors']


In [4]:
# Step 4 — State contract: every documented field is populated

expected = {
    "parsed_spec_sections",
    "current_op_idx",
    "op_iteration_count",
    "validated_fragments_by_op",
    "validation_errors",
    "final_openapi",
}
assert expected.issubset(out_A.keys()), f"Missing: {expected - set(out_A.keys())}"

assert out_A["current_op_idx"] == 0
assert out_A["op_iteration_count"] == 0
assert out_A["validated_fragments_by_op"] == {}
assert out_A["validation_errors"] == []
logger.info("Loop counters and accumulators OK.")

2026-05-25 21:53:53 [INFO] __main__: Loop counters and accumulators OK.


In [5]:
# Step 5 — Section parsing: shape and monotonically increasing ids
#
# section_id reflects the raw position in the document, NOT the position in
# the kept list. When parse_sections drops a symbolic-title section (e.g.
# the 3GPP cover-page table) its id is skipped — so the kept list may start
# at "1" and have gaps where excluded sections lived. What we guarantee:
#   - every kept entry has the required keys and a numeric section_id
#   - ids are strictly increasing (no duplicates, no reordering)

sections = out_A["parsed_spec_sections"]
assert isinstance(sections, list) and len(sections) > 0

prev_id = -1
for section in sections:
    assert set(section.keys()) >= {"section_id", "title", "content"}
    assert isinstance(section["title"], str) and section["title"].strip()
    current_id = int(section["section_id"])
    assert current_id > prev_id, (
        f"section_id must be strictly increasing; saw {prev_id} → {current_id}"
    )
    prev_id = current_id

sizes = [len(s["content"]) for s in sections]
logger.info(f"Parsed {len(sections)} sections.")
logger.info(f"  First/last id    : {sections[0]['section_id']} … {sections[-1]['section_id']}")
logger.info(f"  Min content size : {min(sizes):,} chars")
logger.info(f"  Max content size : {max(sizes):,} chars")
logger.info(f"  Avg content size : {sum(sizes) // len(sizes):,} chars")

2026-05-25 21:53:53 [INFO] __main__: Parsed 696 sections.
2026-05-25 21:53:53 [INFO] __main__:   First/last id    : 1 … 696
2026-05-25 21:53:53 [INFO] __main__:   Min content size : 2 chars
2026-05-25 21:53:53 [INFO] __main__:   Max content size : 30,180 chars
2026-05-25 21:53:53 [INFO] __main__:   Avg content size : 1,353 chars


In [6]:
# Step 6 — final_openapi must be a deep copy of the legacy (not the same object)

assert legacy_openapi is not None, "Skip Scenario A when no legacy is present."

final = out_A["final_openapi"]
assert final is not legacy_openapi, "final_openapi must be a *copy*, not the legacy reference"
assert final.get("info") == legacy_openapi.get("info"), "info block must match the legacy"
assert set(final.get("paths") or {}) == set(legacy_openapi.get("paths") or {}), \
    "paths must match the legacy"

# Mutating the copy must not affect the original.
final["info"]["_sentinel"] = True
assert "_sentinel" not in (legacy_openapi.get("info") or {})
del final["info"]["_sentinel"]

logger.info("final_openapi correctly seeded as a deep copy of the legacy.")
logger.info(f"  info.title : {final['info'].get('title')!r}")
logger.info(f"  paths      : {len(final.get('paths') or {})}")
logger.info(f"  schemas    : {len((final.get('components') or {}).get('schemas') or {})}")

2026-05-25 21:53:53 [INFO] __main__: final_openapi correctly seeded as a deep copy of the legacy.
2026-05-25 21:53:53 [INFO] __main__:   info.title : 'Provisioning MnS'
2026-05-25 21:53:53 [INFO] __main__:   paths      : 1
2026-05-25 21:53:53 [INFO] __main__:   schemas    : 15


In [7]:
# Step 7 — rules_bank is forwarded untouched (downstream iterates rules_bank['rules'])
# Loader doesn't return rules_bank explicitly because LangGraph state already
# carries it; here we just sanity-check the input we passed is iterable rule-by-rule.

rules = rules_bank.get("rules") or []
assert isinstance(rules, list)
logger.info(f"rules_bank['rules'] has {len(rules)} entries, iterable as a flat list.")
if rules:
    logger.info(f"  First rule keys: {sorted(rules[0].keys())}")

2026-05-25 21:53:53 [INFO] __main__: rules_bank['rules'] has 240 entries, iterable as a flat list.
2026-05-25 21:53:53 [INFO] __main__:   First rule keys: ['discard_suggestion', 'openapi_mapping', 'reflection_confidence', 'reflection_flagged', 'reflection_reasoning', 'rule_text', 'rule_type', 'section_id', 'section_title', 'source_name', 'split_suggestion', 'validation_notes', 'validation_passed']


## Scenario B — no legacy (spec + rules only)

Loader must:
1. Parse 3GPP sections (same as A).
2. Seed `final_openapi` from the **empty skeleton**, with `info` populated from `rules_bank.metadata`.
3. Initialize counters.

In [8]:
# Step 8 — Run loader without legacy_openapi

state_B = {
    "spec_doc_path": str(SPEC_PATH),
    "rules_bank": rules_bank,
    "legacy_openapi": None,
}
out_B = loader_node(state_B)

final_B = out_B["final_openapi"]
assert final_B["openapi"] == "3.0.3"
assert final_B["paths"] == {}
assert final_B["components"] == {"schemas": {}}

# info should reflect rules_bank.metadata (title placeholder + provenance markers).
assert isinstance(final_B["info"], dict)
logger.info(f"final_openapi.info (no legacy): {final_B['info']}")

2026-05-25 21:53:53 [INFO] openapi_generator.nodes.loader: Loader → parsed 696 sections from 28532-i00.md (excluded 1 symbolic-title section(s))
2026-05-25 21:53:53 [INFO] openapi_generator.nodes.loader: Loader → rules_bank: 240 rule(s); legacy_openapi: absent
2026-05-25 21:53:53 [INFO] openapi_generator.nodes.loader: Loader → no legacy provided; starting from empty skeleton (info from rules_bank: True)
2026-05-25 21:53:53 [INFO] __main__: final_openapi.info (no legacy): {'title': 'OpenAPI generated from 28532-i00', 'x-rules-bank-generated-at': '2026-04-27T23:11:13.353089-03:00', 'x-rules-bank-model': 'gpt-4.1-mini'}


In [9]:
# Step 9 — Sanity: sections parsed the same way regardless of legacy presence

assert len(out_B["parsed_spec_sections"]) == len(out_A["parsed_spec_sections"])
logger.info("Section parsing is independent of legacy presence. OK.")

2026-05-25 21:53:53 [INFO] __main__: Section parsing is independent of legacy presence. OK.


## Failure-mode tests

Loader must degrade gracefully when the spec path is missing or invalid — return an empty section list, log a warning, never raise. final_openapi still gets seeded from whatever rules/legacy were passed.

In [10]:
# Step 10 — Nonexistent spec path → empty sections, no crash

out_missing = loader_node({
    "spec_doc_path": "/nonexistent/path/spec.md",
    "rules_bank": rules_bank,
    "legacy_openapi": legacy_openapi,
})
assert out_missing["parsed_spec_sections"] == []
assert out_missing["current_op_idx"] == 0
# final_openapi still seeded from legacy when present:
if legacy_openapi:
    assert out_missing["final_openapi"].get("paths") == legacy_openapi.get("paths")
logger.info("Missing spec path → graceful degradation OK.")

2026-05-25 21:53:53 [WARNING] openapi_generator.nodes.loader: Loader → spec_doc_path does not exist: /nonexistent/path/spec.md
2026-05-25 21:53:53 [INFO] openapi_generator.nodes.loader: Loader → rules_bank: 240 rule(s); legacy_openapi: present
2026-05-25 21:53:53 [INFO] openapi_generator.nodes.loader: Loader → seeding final_openapi from legacy (1 path(s), 15 schema(s))
2026-05-25 21:53:53 [INFO] __main__: Missing spec path → graceful degradation OK.


In [11]:
# Step 11 — No inputs at all → empty everything, no crash

out_blank = loader_node({})
assert out_blank["parsed_spec_sections"] == []
assert out_blank["current_op_idx"] == 0
assert out_blank["final_openapi"]["openapi"] == "3.0.3"
assert out_blank["final_openapi"]["info"] == {}
assert out_blank["final_openapi"]["paths"] == {}
logger.info("Empty state → empty skeleton, no crash. OK.")

2026-05-25 21:53:53 [WARNING] openapi_generator.nodes.loader: Loader → no spec_doc_path provided
2026-05-25 21:53:53 [INFO] openapi_generator.nodes.loader: Loader → rules_bank: 0 rule(s); legacy_openapi: absent
2026-05-25 21:53:53 [INFO] openapi_generator.nodes.loader: Loader → no legacy provided; starting from empty skeleton (info from rules_bank: False)
2026-05-25 21:53:53 [INFO] __main__: Empty state → empty skeleton, no crash. OK.


In [12]:
# Step 12 — DI uniformity: loader must accept llm/retriever kwargs and ignore them
# (build_openapi_gen_graph injects the same kwargs into every node via partial)

sentinel_llm = object()
sentinel_ret = object()
out_di = loader_node(
    {"spec_doc_path": str(SPEC_PATH), "rules_bank": rules_bank, "legacy_openapi": None},
    llm=sentinel_llm,
    retriever=sentinel_ret,
)
assert len(out_di["parsed_spec_sections"]) > 0
logger.info("Loader accepts llm/retriever kwargs without using them — DI contract OK.")

2026-05-25 21:53:53 [INFO] openapi_generator.nodes.loader: Loader → parsed 696 sections from 28532-i00.md (excluded 1 symbolic-title section(s))
2026-05-25 21:53:53 [INFO] openapi_generator.nodes.loader: Loader → rules_bank: 240 rule(s); legacy_openapi: absent
2026-05-25 21:53:53 [INFO] openapi_generator.nodes.loader: Loader → no legacy provided; starting from empty skeleton (info from rules_bank: True)
2026-05-25 21:53:53 [INFO] __main__: Loader accepts llm/retriever kwargs without using them — DI contract OK.
